# 1. Initial Configurations

In [162]:
# Configuration and library imports
import boto3
import awswrangler as wr
import os

# Analysis libraries
import numpy as np
import pandas as pd
import pyarrow as pa
import json
from datetime import datetime

In [128]:
# # Database configurations
# DATABASE = "db_data_raw_elo"
# OUTPUT_BUCKET = ""
# region_name = "us-east-1"

# session = boto3.Session(profile_name='')
# glue_client = session.client("glue", region_name=region_name)
# s3_resource = session.client('s3',region_name='uregion_name')

# 2. Get data

## List columns and types from Glue

In [129]:
# # Table name
# table_name = "tb_clients"

# original_athena_types = {} # Column types dictionary
# columns_from_glue = []     # Column names list
# select_columns_str = "*"   # Default
# percentual_sample = 0.05   # Get only 5% sample

# try:
#     glue_response = glue_client.get_table(DatabaseName=DATABASE, Name=table_name)
#     glue_columns_meta = glue_response["Table"]["StorageDescriptor"]["Columns"]
#     for c_meta in glue_columns_meta:
#         col_name = c_meta["Name"]
#         col_type = c_meta["Type"].lower() # Needs to be lower case for comparison later
#         columns_from_glue.append(col_name)
#         original_athena_types[col_name] = col_type
    
#     # Needs to be in the same order as in Glue/Athena
#     select_columns_str = ", ".join([f'"{c}"' for c in columns_from_glue]) 
    
#     print(f"Glue Schema. Total columns: {len(columns_from_glue)}")
    
#     print(f"Glue Schema. Total columns: {len(columns_from_glue)}\nTipos originais do Athena/Glue:")
#     for i, (col, dtype) in enumerate(original_athena_types.items()):
#         print(f"  {col}: {dtype}")
# except Exception as e:
#     print(f"Error retrieving schema from Glue.: {e}")

## Code to execute SQL Query

In [130]:
# query_athena = f"""
# SELECT {select_columns_str}
# FROM {DATABASE}.{table_name}
# TABLESAMPLE BERNOULLI({percentual_sample}) 
# """ # Makes a random sample where every line has a percentage of being included

# print(f"\nRunning a query in Athena to sample {percentual_sample}% from table {table_name}.")
# try:
#     athena_response = wr.athena.read_sql_query(
#         boto3_session=session,
#         sql=query_athena,
#         database=DATABASE,
#         s3_output={OUTPUT_BUCKET},
#         ctas_approach=True # Create Table As Select approach for better performance
#         keep_files=False,
#         dtype_backend="pyarrow"
#     )
#     print("Athena sample (rows × columns):", athena_response.shape)

# except Exception as e:
#     print(f"Error executing query in Athena: {e}")
#     athena_response = pd.DataFrame()

# df_sample = athena_response

## Using a test Dataset

In [131]:
cols_canon = ['threeDSCompInd', 'purchaseDate', 'transStatus', 'purchaseAmount']

In [132]:
test_data = pd.read_csv('../bases_de_dados/sintetica/dataset_sintetico.csv', encoding='latin1')

df_sample = test_data

C:\Users\ana.esquinelato\AppData\Local\Temp\ipykernel_20180\305579670.py:1: DtypeWarning: Columns (51,60,61,62,63) have mixed types. Specify dtype option on import or set low_memory=False.
  test_data = pd.read_csv('../bases_de_dados/sintetica/dataset_sintetico.csv', encoding='latin1')


# 3. Data Overview and Descriptive Statistics

_What am I working with?_

To have a deep understanding of the data, to handle it efficiently in future machine learning tasks.
Start by characterizing the data to the number and types of features, missing rate and percentage of duplicate observations.

## Create versions

In [133]:
sample_v1 = df_sample.copy()
# sample_v2 = sample_v1.copy()

curr_version = sample_v1

## Missing Values

### Check rows with all missing values

In [134]:
rows_all_missing = curr_version[curr_version.isnull().all(axis=1)]

print(f'\nFound {len(rows_all_missing)} rows with all values ​​missing.')


Found 0 rows with all values ​​missing.


#### Drop rows with all missing values

In [135]:
# Drops rows with all values missing

curr_version.drop(rows_all_missing.index, inplace=True)

### Check missing values per column

In [136]:
# Creates a Series with count and percentage of missing values per column
missing_values = curr_version.isnull().sum()
missing_percentage = (missing_values / len(curr_version)) * 100
missing_df = pd.DataFrame({'count': missing_values.values, 'percentage': missing_percentage})
missing_df = missing_df[missing_df['count'] > 0].sort_values(by='percentage', ascending=False)

print("Columns with missing values ​​(sorted by %):")
print(missing_df)

# List columns with +80% missing values
limit_nuls = 80
most_missing_values_df = missing_df[missing_df['percentage'] > limit_nuls]
most_missing_values_ls = most_missing_values_df.index.tolist()

Columns with missing values ​​(sorted by %):
                              count  percentage
billAddrLine3                 10000      100.00
shipAddrLine2                 10000      100.00
shipAddrLine3                 10000      100.00
billAddrLine2                 10000      100.00
sellerInfo                     9528       95.28
multiTransaction               9528       95.28
payTokenInfo                   9493       94.93
payTokenInd                    9493       94.93
recurringDate                  9024       90.24
paymentInfo                    9024       90.24
recurringFrequency             9024       90.24
recurringExpiry                9024       90.24
recurringInd                   9024       90.24
recurringCurrency              9024       90.24
recurringAmount                9024       90.24
threeRIInd                     9024       90.24
acctInfo                       9015       90.15
transStatusReason              9015       90.15
threeDSRequestorChallengeInd   8567       8

### Negative or Zero purchaseAmount

In [137]:
# Not all missing values means the same, sometimes they indicate special cases (ex: returns or cancellations)

# Check if values in Purchase_Amount are zero or negative
negative_purchase_amounts = curr_version[curr_version['purchaseAmount'] <= 0]

print(f'\n{str(sum(negative_purchase_amounts['purchaseAmount'] == 0))} found with a value equal to zero and {str(sum(negative_purchase_amounts['purchaseAmount'] < 0))} with negative values ​​in purchaseAmount.')


0 found with a value equal to zero and 0 with negative values ​​in purchaseAmount.


#### Drop values with Zero or Negative purchaseAmount

In [138]:
curr_version.drop(columns=negative_purchase_amounts.index, inplace=True)

print(f'\n{str(sum(negative_purchase_amounts['purchaseAmount'] == 0))} rows deleted with a value equal to zero in purchaseAmount.') 
print(f'\n{str(sum(negative_purchase_amounts['purchaseAmount'] < 0))} rows deleted with negative values ​​in purchaseAmount.')


0 rows deleted with a value equal to zero in purchaseAmount.

0 rows deleted with negative values ​​in purchaseAmount.


#### Drops columns with +80% of missing values

In [139]:
# Delete columns with +80% missing values and updates lists

remove_missing = [col for col in most_missing_values_ls if col not in cols_canon]

curr_version.drop(columns=remove_missing, inplace=True)

print(f'\n{len(remove_missing)} columns excluded because they contain more than {limit_nuls}% of missing values.')


20 columns excluded because they contain more than 80% of missing values.


## Unique Values

### Higher Cardinality

In [140]:
unique_values = curr_version.nunique()
porcentage_uniques = (unique_values / len(curr_version)) * 100

unique_df = pd.DataFrame({'count': porcentage_uniques.values, 'percentage': porcentage_uniques})
unique_df = unique_df[unique_df['count'] > 0].sort_values(by='percentage', ascending=False)

limit_unique = 99
most_unique_values_df = unique_df[unique_df['percentage'] > limit_unique]

print("Columns with the highest unique values:")
print(unique_df.head(15))

Columns with the highest unique values:
                       count  percentage
browserIP             100.00      100.00
threeDSServerTransID  100.00      100.00
merchantDocument      100.00      100.00
cardHolderDocument    100.00      100.00
acctNumber            100.00      100.00
billAddrPostCode       99.95       99.95
shipAddrPostCode       99.89       99.89
purchaseDate           99.86       99.86
shipAddrLine1          99.35       99.35
billAddrLine1          99.31       99.31
email                  92.54       92.54
purchaseAmount         88.05       88.05
browserUserAgent       84.17       84.17
cardholderName         79.32       79.32
shipAddrCity           37.53       37.53


### Low Cardinality

In [141]:
# Columns with only one value
least_distinct = unique_values[unique_values == 1]
least_distinct_values_ls = least_distinct.index.tolist()

print("Columns with only one value:")
print(least_distinct)

Columns with only one value:
messageType         1
messageVersion      1
purchaseExponent    1
threeDSCompInd      1
transType           1
messageExtension    1
dtype: int64


### Drop Unique Values

In [142]:
remove_high_cardinality = [col for col in most_unique_values_df.index if col not in cols_canon]
remove_low_cardinality = [col for col in least_distinct.index if col not in cols_canon]

curr_version.drop(columns=remove_high_cardinality, inplace=True)
curr_version.drop(columns=remove_low_cardinality, inplace=True)

print(f'\n{len(remove_high_cardinality)} columns excluded because they contain {limit_unique}% of unique values.')

print(f'\n{len(remove_low_cardinality)} columns excluded because they contain only one value.')


9 columns excluded because they contain 99% of unique values.

5 columns excluded because they contain only one value.


## Duplicates Values

In [143]:
duplicate_df = curr_version[curr_version.duplicated(keep=False)]

print(f'\nDuplicate rows {len(duplicate_df)} were found in the dataset.')


Duplicate rows 0 were found in the dataset.


### Drops the duplicates value

In [144]:
curr_version.drop_duplicates()

print(f'{len(duplicate_df)} lines with duplicate values ​​have been deleted.')

0 lines with duplicate values ​​have been deleted.


## Manual Analysis

In [145]:
curr_version.drop(columns=
    [
        'messageCategory', # Só tem valor 1 e 8, não sei o que é
        # 'deviceChannel', # Só tem valor 1, 2 ou 3, não faz sentido manter porque existe o browserUserAgent
        'browserTZ', # Talvez pra comparar com o fuso horário do usuário
        # 'browserScreenHeight', # Não faz sentido manter porque tem browserUserAgent
        # 'browserScreenWidth', # Não faz sentido manter porque tem browserUserAgent
        'cardholderName', # Alta cardinalidade, problemas com compliance
        # 'cardExpiryDate', # Talvez porque pode ser usado para fraude
        'merchantName', # Deveria ficar o merchantDocument, por motivos de compliance, mas tem alta cardinalidade também
        'cardSecurityCode', # Alta cardinalidade, problemas com compliance
        # 'eci', # 5 risco do emissor ou 6 risco da bandeira, 7 autenticação não realizada risco do estabelecimento
        # 'challengeWindowSize',
        'billAddrCountry', # Há a possibilidade de uso de VPN pra fraude
        'shipAddrCountry'
    ], inplace=True)

In [146]:
# curr_version['purchaseCurrency'] = curr_version['purchaseCurrency'].astype('int64')
# curr_version['merchantCountryCode'] = curr_version['merchantCountryCode'].astype('category')
curr_version['transStatus'] = curr_version['transStatus'].astype('bool')
# curr_version['eci'] = curr_version['eci'].astype('category')
curr_version['fraud_label'] = curr_version['fraud_label'].astype('bool')
curr_version['addrMatch'] = curr_version['addrMatch'].astype('bool')
# curr_version['purchaseDateBin'] = curr_version['purchaseDateBin'].astype('category')
# curr_version['billLocation'] = curr_version['billLocation'].astype('category')

In [147]:
curr_version.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
productType,10000,3,ONLINE_PURCHASE,6929,NaN,NaN,NaN,NaN,NaN,NaN,NaN
purchaseCurrency,10000.0,NaN,NaN,NaN,971.2928,43.313364,840.0,986.0,986.0,986.0,986.0
merchantCountryCode,10000.0,NaN,NaN,NaN,133.3914,193.138979,76.0,76.0,76.0,76.0,840.0
deviceChannel,10000.0,NaN,NaN,NaN,1.8136,0.419372,1.0,2.0,2.0,2.0,3.0
threeDSCompInd,10000,1,Y,10000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
browserLanguage,10000,3,pt-BR,6916,NaN,NaN,NaN,NaN,NaN,NaN,NaN
purchaseInstalData,10000.0,NaN,NaN,NaN,1.3977,0.659529,1.0,1.0,1.0,2.0,3.0
purchaseDate,10000.0,NaN,NaN,NaN,20251011468991.207031,18797074.266925,20250929191017.0,20251007062551.5,20251014204405.0,20251022030958.0,20251029190209.0
browserScreenHeight,10000.0,NaN,NaN,NaN,1122.0069,277.76469,640.0,883.0,1118.0,1364.0,1599.0
browserScreenWidth,10000.0,NaN,NaN,NaN,1756.9924,459.994493,960.0,1358.0,1755.0,2158.25,2559.0


## Update list of deleted and kept columns

In [148]:
# Create varibles with different types of columns

deleted_cols = []
kept_cols = []
types_cols_kept = []

In [151]:
# Update kept columns list
for col in curr_version.columns:
    kept_cols.append(col)

print(f'\nColunas mantidas:\n{kept_cols}')

# Update column type list after dropping columns using columns as guides
types_cols_kept = curr_version.dtypes

# Update deleted columns list
deleted_cols = list(set(df_sample.columns) - set(curr_version.columns))

print(f'\nExcluded columns:\n{deleted_cols}')

print(f'\n{len(kept_cols)} columns kept. {len(deleted_cols)} columns deleted.')
print(f'\nTypes of kept columns: \n{types_cols_kept}')


Colunas mantidas:
['productType', 'purchaseCurrency', 'merchantCountryCode', 'deviceChannel', 'threeDSCompInd', 'browserLanguage', 'purchaseInstalData', 'purchaseDate', 'browserScreenHeight', 'browserScreenWidth', 'browserUserAgent', 'cardExpiryDate', 'email', 'transStatus', 'eci', 'challengeWindowSize', 'purchaseAmount', 'fraud_label', 'billAddrCity', 'billAddrState', 'shipAddrCity', 'shipAddrState', 'addrMatch']

Excluded columns:
['recurringCurrency', 'acctNumber', 'merchantDocument', 'threeDSServerTransID', 'billAddrLine2', 'merchantName', 'acctInfo', 'shipAddrCountry', 'transStatusReason', 'browserTZ', 'multiTransaction', 'shipAddrLine1', 'billAddrPostCode', 'browserIP', 'messageExtension', 'billAddrLine1', 'shipAddrPostCode', 'recurringDate', 'sellerInfo', 'cardHolderDocument', 'cardholderName', 'shipAddrLine3', 'payTokenInd', 'recurringInd', 'transType', 'recurringFrequency', 'payTokenInfo', 'messageVersion', 'threeRIInd', 'paymentInfo', 'billAddrCountry', 'recurringAmount', 't

## Separation between column types

In [152]:
print("Sample dimensions (rows × columns):", curr_version.shape)
print("\nGeneral information and data types:")
curr_version.info(verbose=True, show_counts=True)

# Separate columns by data type
numeric_cols = []
string_cols = []
datetime_cols = []
date_cols = []
bool_cols = []
other_cols = []

# Classification based on pandas and pyarrow dtypes
for col, dtype in types_cols_kept.items():
    if pd.api.types.is_numeric_dtype(dtype) and not pd.api.types.is_bool_dtype(dtype):
        numeric_cols.append(col)
    elif pd.api.types.is_string_dtype(dtype) or \
         (isinstance(dtype, pd.ArrowDtype) and (pa.types.is_string(dtype.pyarrow_dtype) or pa.types.is_large_string(dtype.pyarrow_dtype))):
        string_cols.append(col)
    elif isinstance(dtype, pd.ArrowDtype) and pa.types.is_date(dtype.pyarrow_dtype):
        date_cols.append(col_name)
    elif pd.api.types.is_datetime64_any_dtype(dtype) or \
         (isinstance(dtype, pd.ArrowDtype) and pa.types.is_timestamp(dtype.pyarrow_dtype)):
        datetime_cols.append(col)
    elif pd.api.types.is_bool_dtype(dtype):
        bool_cols.append(col)
    else:
        other_cols.append(col)

print(f"Numerical columns: {len(numeric_cols)}")
print(f"Text columns: {len(string_cols)}")
print(f"Datetime columns: {len(datetime_cols)}")
print(f"Boolean columns: {len(bool_cols)}")
print(f"Unsorted columns: {len(other_cols)}")

Sample dimensions (rows × columns): (10000, 23)

General information and data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   productType          10000 non-null  object 
 1   purchaseCurrency     10000 non-null  int64  
 2   merchantCountryCode  10000 non-null  int64  
 3   deviceChannel        10000 non-null  int64  
 4   threeDSCompInd       10000 non-null  object 
 5   browserLanguage      10000 non-null  object 
 6   purchaseInstalData   10000 non-null  int64  
 7   purchaseDate         10000 non-null  int64  
 8   browserScreenHeight  10000 non-null  int64  
 9   browserScreenWidth   10000 non-null  int64  
 10  browserUserAgent     10000 non-null  object 
 11  cardExpiryDate       10000 non-null  int64  
 12  email                10000 non-null  object 
 13  transStatus          10000 non-null  bool   
 14  ec

## Save in parquet

In [167]:
# save_s3 = f"s3://elo-fraud-detection-data-test2/{datetime.now().strftime("%Y%m%d")}/data_cleaned.parquet"
# print(f"Saving a sample of Athena in Parquet to:{save_s3}")
# wr.s3.to_parquet(
#     df=curr_version,
#     path=save_s3,
#     index=False,
#     dataset=True,
#     mode="overwrite"
# )
# print("\nSample of Athena saved in Parquet.")

In [ ]:
def create_path(extension):
    folder = f'{datetime.now().strftime('%Y%m%d')}'
    os.makedirs(folder, exist_ok=True)
    return f'{folder}/df_cleaned.{extension}'

# Save as Parquet
curr_version.to_parquet(create_path('parquet'), engine="pyarrow", index=False)

# Save as CSV
curr_version.to_csv(create_path('csv'), index=False)